# Idiology and polarization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import seaborn as sns

from tqdm import tqdm
import pickle

import os

import matplotlib.cm as cm
import matplotlib.colors as colrs
from matplotlib.patches import PathPatch
from matplotlib.path import Path


from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import r2_score

import folium
import json
import mapply

from pyproj import Proj, transform
from matplotlib.path import Path
import shapely
from shapely.geometry import Point, mapping
from shapely.geometry import shape as Shape
from shapely.ops import transform as Shapely_transform

import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import pdist

from statsmodels.stats.outliers_influence import variance_inflation_factor

from scipy.stats.stats import pearsonr
import scipy.stats as stats
from pyproj import Transformer

In [ ]:
def PolygonPatch(polygon, **kwargs):
    """Replaces descartes.PolygonPatch for Shapely 2.0+"""
    # Get the exterior path codes and coordinates
    ext_coords = np.asarray(polygon.exterior.coords)
    codes = [Path.MOVETO] + [Path.LINETO] * (len(ext_coords) - 2) + [Path.CLOSEPOLY]
    vertices = ext_coords

    # Add any interior holes if they exist
    for interior in polygon.interiors:
        int_coords = np.asarray(interior.coords)
        int_codes = (
            [Path.MOVETO] + [Path.LINETO] * (len(int_coords) - 2) + [Path.CLOSEPOLY]
        )
        vertices = np.concatenate([vertices, int_coords])
        codes.extend(int_codes)

    path = Path(vertices, codes)
    return PathPatch(path, **kwargs)


def PolygonPatch(polygon, **kwargs):
    """Replaces descartes.PolygonPatch for Shapely 2.0+ (Handles MultiPolygons)"""
    # If it's a MultiPolygon, break it down into individual Polygons
    if hasattr(polygon, "geoms"):
        polygons = polygon.geoms
    else:
        polygons = [polygon]

    vertices_list = []
    codes_list = []

    for poly in polygons:
        # Process individual polygon exterior
        ext_coords = np.asarray(poly.exterior.coords)
        ext_codes = (
            [Path.MOVETO] + [Path.LINETO] * (len(ext_coords) - 2) + [Path.CLOSEPOLY]
        )

        vertices_list.append(ext_coords)
        codes_list.extend(ext_codes)

        # Process individual polygon interior holes
        for interior in poly.interiors:
            int_coords = np.asarray(interior.coords)
            int_codes = (
                [Path.MOVETO] + [Path.LINETO] * (len(int_coords) - 2) + [Path.CLOSEPOLY]
            )

            vertices_list.append(int_coords)
            codes_list.extend(int_codes)

    # Combine everything into a single Matplotlib path
    if vertices_list:
        vertices = np.concatenate(vertices_list)
        path = Path(vertices, codes_list)
        return PathPatch(path, **kwargs)
    else:
        raise ValueError("Provided geometry contains no valid polygons.")

In [ ]:
# Font
font_size = 16
fz = 1.5

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["CMU Serif Roman"] + plt.rcParams["font.serif"]
plt.rcParams["font.size"] = 18

mapply.init(n_workers=37, chunk_size=1, progressbar=True)


lambert93 = "epsg:2154"  # current version & updated
ntf = "epsg:27572"  # old version & outdated
wgs = "epsg:4326"

wgs_to_lambert93 = Transformer.from_crs(wgs, lambert93).transform
wgs_to_ntf = Transformer.from_crs(wgs, ntf).transform
ntf_to_wgs = Transformer.from_crs(ntf, wgs).transform
lambert93_to_wgs = Transformer.from_crs(lambert93, wgs).transform

In [ ]:
import os
WORKING_DIR = os.environ.get("REPO_ROOT", os.path.abspath(".."))  # repo root (notebooks run from notebooks/)
DATA_DIR = f"{WORKING_DIR}/data"
IMG_DIR = f"{WORKING_DIR}/images"

## Load data

### France shape

In [ ]:
france_file = f"{DATA_DIR}/france_shape/france.geojson"

fd = open(france_file, "r")
geojson_content = json.load(fd)
fd.close()

france_mainland = Shape(geojson_content["features"][0]["geometry"])
france_mainland = france_mainland.buffer(0)
# Changing to lat, lon
france_shape = france_mainland
france_shape

### Cities

In [ ]:
cities_names = ["Paris", "Lyon", "Bordeaux", "Toulouse", "Nantes", "Lille"]


cities = {}

for city_name in cities_names:
    print(f"Loading {city_name} shape...")
    city_file = f"{DATA_DIR}/cities/{city_name}.geojson"

    fd = open(city_file, "r")
    geojson_content = json.load(fd)
    fd.close()

    cities[city_name] = Shape(geojson_content["features"][0]["geometry"])
    cities[city_name] = cities[city_name].buffer(0)
    # Changing to lat, lon
    # cities[city_name] = Shapely_transform(lambda x, y: (y, x), cities[city_name])
    cities[city_name]

cities["Paris"]

In [ ]:

# ille_france_shape = Shape(geojson_content['features'][0]['geometry'])
# ille_france_shape = ille_france_shape.buffer(0)
# # Changing to lat, lon
# ille_france_shape = Shapely_transform(lambda x, y: (y, x), ille_france_shape)
# ille_france_shape

### Communes

In [ ]:
df_communes = pd.read_pickle(f"{DATA_DIR}/carto/communes.pkl")
print(f"Number of communes: {len(df_communes)}")
df_communes.head(2)

### Load elections

In [ ]:
elections = pickle.load(open(f"{DATA_DIR}/elections/elections_communes.pkl", "rb"))

### Social economical indicators

In [ ]:
df_social_economic_2019 = pd.read_pickle(
    f"{DATA_DIR}/dossier/df_social_economical_2019.pkl"
)
df_social_economic_2024 = pd.read_pickle(
    f"{DATA_DIR}/dossier/df_social_economical_2024.pkl"
)

print(
    f"Number of communes with social-economic data in 2019 dataset: {len(df_social_economic_2019)}"
)
print(
    f"Number of communes with social-economic data in 2024 dataset: {len(df_social_economic_2024)}"
)
df_social_economic_2024.head(2)

### Apps

In [ ]:



df_european_parlamentary_election_traffic_2019 = pd.read_pickle(
    f"{DATA_DIR}/traffic/df_traffic_commune_apps_with_rca_europe_2019_20_07.pkl"
)
df_european_parlamentary_election_traffic_2024 = pd.read_pickle(
    f"{DATA_DIR}/traffic/df_traffic_commune_apps_with_rca_europe_2024_20_07.pkl"
)

print(
    f"Number of communes with traffic data in 2019 dataset: {len(df_european_parlamentary_election_traffic_2019)}"
)
print(
    f"Number of communes with traffic data in 2024 dataset: {len(df_european_parlamentary_election_traffic_2024)}"
)

#### Selected Apps

In [ ]:
selected_apps_2019 = [
    "Amazon Prime Video",
    "Apple Music",
    "Apple Video",
    "Apple iMessage",
    "CanalPlus",
    "DailyMotion",
    "Deezer",
    "Facebook",
    "Google News",
    "Instagram",
    "LinkedIn",
    "Molotov TV",
    "Netflix",
    "NewsMag",
    "NewsPaper",
    "SnapChat",
    "Sports News",
    "Spotify",
    "Starz",
    "TV5MONDE",
    "Telegram",
    "Twitch",
    "Twitter",
    "Vine",
    "WhatsApp",
    "Wikipedia",
    "Youtube",
]

selected_apps_2024 = [
    "Amazon Prime Video",
    "Apple Music",
    "Apple TV",
    "Apple Video",
    "Apple iMessage",
    "CanalPlus",
    "DailyMotion",
    "Deezer",
    "Discord",
    "Disney+",
    "Facebook",
    "Google News",
    "Instagram",
    "LinkedIn",
    "Molotov TV",
    "Netflix",
    "NewsMag",
    "NewsPaper",
    "Pluto TV",
    "Signal",
    "SnapChat",
    "Sports News",
    "Spotify",
    "Starz",
    "TV5MONDE",
    "Telegram",
    "TikTok",
    "Twitch",
    "Twitter",
    "Vine",
    "WhatsApp",
    "Wikipedia",
    "Youtube",
]

print(f"The total number of apps in 2019 to be consider is {len(selected_apps_2019)}")
print(f"The total number of apps in 2024 to be consider is {len(selected_apps_2024)}")

### Remove apps with weird time series

In [ ]:
apps_to_del = [
    "Apple Siri",
    "Adult Content",
    "Apple Game Center",
    "Apple Maps",
    "Apple iTunes",
    "Apple Private Relay",
    "DownloadsWeb",
    "Foursquare",
    "Glovo",
    "Google Assistant",
    "IM",
    "Kayak",
    "Microsoft",
    "Pandora",
    "Pet Rescue Saga",
    "Starz",
    "TeamViewer",
    "TinyCam",
    "TeamViewer",
    "TinyCam",
    "Tor",
    "Unreal Engine games",
    "Vine",
]

In [ ]:
selected_apps_2019 = set(selected_apps_2019) - set(apps_to_del)
selected_apps_2019 = sorted(list(selected_apps_2019))

selected_apps_2024 = set(selected_apps_2024) - set(apps_to_del)
selected_apps_2024 = sorted(list(selected_apps_2024))

selected_columns_2019 = (
    ["insee"]
    + list(selected_apps_2019)
    + [f"{app}_rca" for app in selected_apps_2019]
    + [f"{app}_srca" for app in selected_apps_2019]
)
selected_columns_2024 = (
    ["insee"]
    + list(selected_apps_2024)
    + [f"{app}_rca" for app in selected_apps_2024]
    + [f"{app}_srca" for app in selected_apps_2024]
)

df_european_parlamentary_election_traffic_2019 = (
    df_european_parlamentary_election_traffic_2019[selected_columns_2019]
)
df_european_parlamentary_election_traffic_2024 = (
    df_european_parlamentary_election_traffic_2024[selected_columns_2024]
)

elections["europe_2019"]["traffic"] = df_european_parlamentary_election_traffic_2019
elections["europe_2024"]["traffic"] = df_european_parlamentary_election_traffic_2024

## Merge

In [ ]:
for election in elections:
    election_year = elections[election]["date"].year
    election_name = elections[election]["name"]

    df_election = elections[election]["results"]
    df_traffic = elections[election]["traffic"]

    if election_year == 2019:
        df_social_economic = df_social_economic_2019
    elif election_year == 2024:
        df_social_economic = df_social_economic_2024
    else:
        raise ValueError("Year not found")

    df_communes_elections = df_communes.merge(df_election, on="insee", how="inner")
    df_communes_elections_traffic = df_communes_elections.merge(
        df_traffic, on="insee", how="inner"
    )
    df_communes_elections_traffic_social_economic = df_communes_elections_traffic.merge(
        df_social_economic, on="insee", how="inner"
    )

    print(f"For election {election_name} ({election_year}):")
    print(
        f"Number of communes with all datasets: {df_communes_elections_traffic_social_economic.shape[0]}"
    )
    df_communes_elections_traffic_social_economic.dropna(inplace=True)
    print(
        f"Number of communes after dropping NaN values: {df_communes_elections_traffic_social_economic.shape[0]}"
    )
    df_communes_elections_traffic_social_economic.head(2)

    elections[election][
        "df_communes_elections_traffic_social"
    ] = df_communes_elections_traffic_social_economic
    print("\n")

In [ ]:
social_media = [
    "Facebook_srca",
    "Instagram_srca",
    "LinkedIn_srca",
    "SnapChat_srca",
    "Twitter_srca",
    "Twitch_srca",
    "TikTok_srca",
]

news = [
    "NewsPaper_srca",
    "Sports News_srca",
    "DailyMotion_srca",
    "NewsMag_srca",
    "Google News_srca",
    "TV5MONDE_srca",
]


messaging = [
    "WhatsApp_srca",
    "Apple iMessage_srca",
    "Signal_srca",
    "Discord_srca",
    "Telegram_srca",
]

streamming = [
    "Youtube_srca",
    "Spotify_srca",
    "CanalPlus_srca",
    "Netflix_srca",
    "Apple Music_srca",
    "Disney+_srca",
    "Apple Video_srca",
    "Molotov TV_srca",
    "Pluto TV_srca",
]

selected_rows = social_media + news + messaging + streamming
selected_rows = [column.replace("_srca", "") for column in selected_rows]

In [ ]:
for election in elections:
    df_communes_elections_traffic_social = elections[election][
        "df_communes_elections_traffic_social"
    ]

    selected_rows_ = set(selected_rows) & set(
        df_communes_elections_traffic_social.columns
    )
    selected_rows_ = list(selected_rows_)
    traffic_france = df_communes_elections_traffic_social[list(selected_rows_)]

    df_rural = df_communes_elections_traffic_social[
        df_communes_elections_traffic_social["urbanization_level"] == "rural"
    ]
    df_rural = df_rural[selected_rows_]
    print(
        f'number of rural communes for election {election} ({elections[election]["date"].year}): {len(df_rural)}'
    )

    df_sub_urban = df_communes_elections_traffic_social[
        df_communes_elections_traffic_social["urbanization_level"] != "rural"
    ]
    df_sub_urban = df_sub_urban[selected_rows_]
    print(
        f'number of urban/sub-urban communes for election {election} ({elections[election]["date"].year}): {len(df_sub_urban)}'
    )

In [ ]:

# all_tiktok/all_newspaper* 100

In [ ]:
for election in elections:

    df_election_eligible_voters = elections[election][
        "df_communes_elections_traffic_social"
    ]
    total_inscrits = df_election_eligible_voters["inscrits"].sum()
    total_voters = df_election_eligible_voters["voters"].sum()

    df_election_eligible_voters_urban = df_election_eligible_voters[
        df_election_eligible_voters["urbanization_level"] != "rural"
    ]
    total_inscrits_urban = df_election_eligible_voters_urban["inscrits"].sum()
    total_voters_urban = df_election_eligible_voters_urban["voters"].sum()

    percentage_urban_inscrits = total_inscrits_urban / total_inscrits * 100
    percentage_urban_voters = total_voters_urban / total_voters * 100
    print(
        f"Election: {election} | Percentage urban inscrits: {percentage_urban_inscrits:.4f} | Percentage urban voters: {percentage_urban_voters:.4f}"
    )

## Maps

In [ ]:
def plot_communes_indicator(
    year,
    variable_name,
    shapes,
    values,
    region,
    vmin,
    vcenter,
    vmax,
    colorbar_xticks,
    title,
    cmap,
    filename,
    norm=None,
    show_region=True,
    show_borders=False,
    show_colorbar=True,
    cities=None,
):

    bbox = region.bounds
    min_x, min_y, max_x, max_y = bbox
    ratio = (max_y - min_y) / (max_x - min_x)

    dim_width = 6
    dim_height = 6 * 1.5 * ratio

    my_cmap = matplotlib.colormaps[cmap]

    my_norm = colrs.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    if norm is not None:
        my_norm = norm

    fig = plt.figure(figsize=(dim_width, dim_height))

    ax = fig.add_axes([0, 0, 1, 0.95])
    ax.set_rasterized(True)

    if show_region:
        patch = PolygonPatch(region, fc="white", ec="tab:grey", zorder=-1, alpha=0.25)
        ax.add_patch(patch)

    for shape, value in zip(shapes, values):
        color = colrs.to_hex(my_cmap(my_norm(value)))
        shape = Shapely_transform(lambda x, y: (y, x), shape)

        patch = PolygonPatch(shape, fc=color, ec="lightgrey" if show_borders else color)
        ax.add_patch(patch)

    plt.autoscale()
    bounds = region.buffer(0.01).bounds
    plt.xlim(bounds[0], bounds[2])
    plt.ylim(bounds[1], bounds[3])
    plt.axis("off")

    if cities is not None:

        for cities_name in cities:
            city_shape = cities[cities_name]

            city_center = city_shape.centroid
            # add a empty circle
            circle = plt.Circle(
                (city_center.x, city_center.y),
                0.2,
                color="tab:grey",
                fill=False,
                zorder=3,
                lw=1.5,
            )
            ax.add_patch(circle)


    if show_colorbar:
        ax = fig.add_axes([0.4, 1.01, 0.5, 0.02])
        sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
        sm.set_array([])

        clb = plt.colorbar(sm, cax=ax, orientation="horizontal")
        clb.ax.set_xticks([vmin, vcenter, vmax])
        clb.ax.set_xticklabels(colorbar_xticks)

        clb.ax.set_title(title, fontsize=18)
        clb.ax.xaxis.set_ticks_position("default")
        clb.ax.tick_params(labelsize=16)

    folder = f"{IMG_DIR}/maps/{year}/{variable_name}"
    os.makedirs(folder, exist_ok=True)

    plt.savefig(f"{folder}/{filename}.pdf", bbox_inches="tight", dpi=150)

    plt.show()

In [ ]:
def plot_standalone_colorbar(
    year,
    variable_name,
    vmin,
    vcenter,
    vmax,
    colorbar_xticks,
    title,
    cmap,
    filename,
    norm=None,
):

    my_cmap = matplotlib.colormaps[cmap]
    my_norm = colrs.TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)
    if norm is not None:
        my_norm = norm

    fig = plt.figure(figsize=(4, 0.25))

    ax = fig.add_axes([0.0, 0, 1, 1])
    sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
    sm.set_array([])

    clb = plt.colorbar(sm, cax=ax, orientation="horizontal")
    clb.ax.set_xticks([vmin, vcenter, vmax])
    clb.ax.set_xticklabels(colorbar_xticks)

    clb.ax.set_title(title, fontsize=18)
    clb.ax.xaxis.set_ticks_position("default")
    clb.ax.tick_params(labelsize=16)

    folder = f"{IMG_DIR}/maps/{year}/{variable_name}"
    os.makedirs(folder, exist_ok=True)

    plt.savefig(f"{folder}/{filename}_colorbar.pdf", bbox_inches="tight", dpi=200)

    plt.show()

In [ ]:
YEAR = 2019
YEAR = 2024

NATIONWIDE = False

In [ ]:
election = f"europe_{YEAR}"
election_name = elections[election]["name"]
df_election = elections[election]["df_communes_elections_traffic_social"]

if NATIONWIDE:
    df_selected_communes = df_election.copy()
    filename_prefix = "nationwide"
else:
    df_selected_communes = df_election[
        df_election["urbanization_level"] != "rural"
    ].copy()
    filename_prefix = "urban_suburban"

df_selected_communes["area"] = df_selected_communes.apply(
    lambda row: Shapely_transform(wgs_to_lambert93, row["geometry"]).area / 1e6, axis=1
)

### Votes

### Socioeconomic indicators

In [ ]:
pop = df_selected_communes["pop"]
density = pop / df_selected_communes["area"]
df_selected_communes["population_density"] = list(density)

df_selected_communes["unemployment_rate"] = (
    df_selected_communes["unemployment_ratio"] * 100
)

In [ ]:
incomes = list(df_selected_communes["median_income"])

incomes_transform, lambda_ = stats.boxcox(incomes)
df_selected_communes["median_income_transform"] = incomes_transform

boxcox_transform = lambda x: (x**lambda_ - 1) / lambda_
boxcox_inverse = lambda x: (x * lambda_ + 1) ** (1 / lambda_)


print(f"Lambda value is {lambda_}")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
ax = axs[0]
bins = np.linspace(min(incomes), max(incomes), 50)
ax.hist(incomes, bins=bins)

ax = axs[1]
bins = np.linspace(min(incomes_transform), max(incomes_transform), 50)
ax.hist(incomes_transform, bins=bins)

plt.show()


a, b, c = (
    np.percentile(incomes_transform, 1),
    np.percentile(incomes_transform, 50),
    np.percentile(incomes_transform, 99),
)

print(f"1st percentile: {a} \n50th percentile: {b} \n99th percentile: {c}")


steps = [18000, 25000, 50000]
vmin, vcenter, vmax = [boxcox_transform(step) for step in steps]
print(f"vmin: {vmin}\nvcenter: {vcenter}\nvmax: {vmax}")

In [ ]:
(50 + 18) / 2

### Apps

In [ ]:
app_1 = "Facebook"
app_2 = "WhatsApp"

fig = plt.figure(figsize=(8, 6))
raw_bins = np.logspace(2.5, 8.5, 100)
xticks = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8]
xticks_label = [
    f"$10^n$" if i == 0 else f"$10^{{n+{i}}}$" for i, x in enumerate(xticks)
]


plt.hist(
    df_selected_communes[app_1], bins=raw_bins, alpha=1, color="tab:blue", label=app_1
)
plt.hist(
    df_selected_communes[app_2],
    bins=raw_bins,
    alpha=0.75,
    color="tab:orange",
    label=app_2,
)

plt.xscale("log")
plt.ylabel("Number of communes")
plt.xlabel(f"Raw traffic")
plt.xlim(raw_bins[0], raw_bins[-1])
plt.xticks(xticks, xticks_label)

plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_raw_{YEAR}.png", bbox_inches="tight", dpi=200
)
plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_raw_{YEAR}.pdf", bbox_inches="tight", dpi=200
)
plt.show()


fig = plt.figure(figsize=(8, 6))
rca_bins = np.linspace(0, 7, 100)
rca_bins_sub = np.linspace(3, 12, 100)
plt.hist(df_selected_communes[f"{app_1}_rca"], bins=rca_bins, alpha=1, color="tab:blue")
plt.hist(
    df_selected_communes[f"{app_2}_rca"], bins=rca_bins, alpha=0.75, color="tab:orange"
)
plt.ylabel("Number of communes")
plt.xlabel(f"RCA Traffic")
plt.xlim(rca_bins[0], rca_bins[-1])
ax = fig.add_axes([0.42, 0.35, 0.45, 0.44])
ax.hist(
    df_selected_communes[f"{app_1}_rca"], bins=rca_bins_sub, alpha=1, color="tab:blue"
)
ax.hist(
    df_selected_communes[f"{app_2}_rca"],
    bins=rca_bins_sub,
    alpha=0.75,
    color="tab:orange",
)
ax.set_xlim(rca_bins_sub[0], rca_bins_sub[-1])
ax.set_ylim(0, 20)

plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_rca_{YEAR}.png", bbox_inches="tight", dpi=200
)
plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_rca_{YEAR}.pdf", bbox_inches="tight", dpi=200
)
plt.show()


fig = plt.figure(figsize=(8, 6))
srca = np.linspace(-1, 1, 100)
plt.hist(df_selected_communes[f"{app_1}_srca"], bins=srca, alpha=1, color="tab:blue")
plt.hist(
    df_selected_communes[f"{app_2}_srca"], bins=srca, alpha=0.75, color="tab:orange"
)

plt.ylabel("Number of communes")
plt.xlabel(f"SRCA Traffic")
plt.xticks([-1, -0.5, 0, 0.5, 1])
plt.xlim(-1, 1)

plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_srca_{YEAR}.png",
    bbox_inches="tight",
    dpi=200,
    transparent=True,
)
plt.savefig(
    f"{IMG_DIR}/rca/{app_1}_{app_2}_srca_{YEAR}.pdf",
    bbox_inches="tight",
    dpi=200,
    transparent=True,
)
plt.show()


fig = plt.figure(figsize=(8, 1))
plt.fill_between([], [], label=f"{app_1}")
plt.fill_between([], [], label=f"{app_2}")

plt.axis("off")

plt.legend(loc="upper right", ncol=2, fancybox=True, frameon=False)

plt.savefig(f"{IMG_DIR}/rca/legend.png", bbox_inches="tight", dpi=200)
plt.savefig(f"{IMG_DIR}/rca/legend.pdf", bbox_inches="tight", dpi=200)
plt.show()

### Cloud of points

In [ ]:
def plot_cloud_of_point(votes, variable, xlabel, xmin, xmax, cmap, filename):








    correlation = np.corrcoef(votes, variable)[0, 1]
    print(f"Correlation between {xlabel} and votes is {correlation:.2f}")

    plt.figure(figsize=(10, 8))
    x_bins = np.linspace(xmin, xmax, 40)
    y_bins = np.linspace(0, 80, 40)

    H, _, _ = np.histogram2d(variable, votes, bins=(x_bins, y_bins))

    my_cmap = cm.get_cmap(cmap)
    my_cmap.set_under("white")
    norm = colrs.Normalize(vmin=1, vmax=0.9 * H.max())

    plt.hist2d(variable, votes, bins=(x_bins, y_bins), cmap=my_cmap, norm=norm)
    plt.colorbar(format=lambda x, _: f"{int(x)}")

    plt.ylabel("Votes\nRassemblement National", fontsize=18)
    plt.yticks(fontsize=18)

    plt.xlabel(xlabel, fontsize=18)
    if xlabel == "Median Income":
        plt.xticks(fontsize=18, rotation=45)
    else:
        plt.xticks(fontsize=18)
    plt.savefig(f"{IMG_DIR}/cloud_points/{filename}.pdf", bbox_inches="tight", dpi=200)
    plt.show()

#### All France

In [ ]:
for election in elections:
    if election == "legislative_2024":
        continue

    df_election = elections[election]["df_communes_elections_traffic_social"]
    df_election_urban = df_election[df_election["urbanization_level"] != "rural"]

    total_population = df_election["pop"].sum()
    total_population_urban = df_election_urban["pop"].sum()
    ratio_urban_population = total_population_urban / total_population

    n_communes = df_election.shape[0]
    n_communes_urban = df_election_urban.shape[0]
    ratio_urban_communes = n_communes_urban / n_communes

    print(
        f"Ratio of urban & suburban population for {election} is {ratio_urban_population:.2f}"
    )
    print(f"The number of communes for {election} is {n_communes}")
    print(
        f"The number of urban & suburban communes for {election} is {n_communes_urban}"
    )
    print("\n")

## Standarize variables

In [ ]:
def standarize_variable(values):
    # standardize by dividing by the standard deviation
    values = np.array(values)
    mean_values = np.mean(values)

    scale = 1 / len(values) * np.sum((values - mean_values) ** 2)
    scale = np.sqrt(scale)

    values_scaled = values / scale
    return values_scaled

###  Median income

In [ ]:
for election in elections:
    df_communes_elections_traffic_social = elections[election][
        "df_communes_elections_traffic_social"
    ]
    election_year = elections[election]["date"].year

    # Median Income
    df_communes_elections_traffic_social["median_income"] = standarize_variable(
        df_communes_elections_traffic_social["median_income"]
    )
    # Unemployment Ratio
    df_communes_elections_traffic_social["unemployment_ratio"] = standarize_variable(
        df_communes_elections_traffic_social["unemployment_ratio"]
    )

    age_columns = [
        "pop_0_14",
        "pop_15_29",
        "pop_30_44",
        "pop_45_59",
        "pop_60_74",
        "pop_75_89",
        "pop_90",
    ]
    for age_column in age_columns:
        df_communes_elections_traffic_social[age_column] = standarize_variable(
            df_communes_elections_traffic_social[age_column]
        )

    if election_year == 2019:
        selected_apps = selected_apps_2019
    elif election_year == 2024:
        selected_apps = selected_apps_2024
    else:
        raise ValueError("Year not found")

    # apps
    for app in selected_apps:
        df_communes_elections_traffic_social[app] = standarize_variable(
            df_communes_elections_traffic_social[app]
        )
        df_communes_elections_traffic_social[f"{app}_rca"] = standarize_variable(
            df_communes_elections_traffic_social[f"{app}_rca"]
        )
        df_communes_elections_traffic_social[f"{app}_srca"] = standarize_variable(
            df_communes_elections_traffic_social[f"{app}_srca"]
        )

    elections[election][
        "df_communes_elections_traffic_social"
    ] = df_communes_elections_traffic_social.copy()

In [ ]:
df_communes_elections_traffic_social_2019 = elections["europe_2024"][
    "df_communes_elections_traffic_social"
]

plt.hist(df_communes_elections_traffic_social_2019["Facebook_srca"], bins=20)
plt.hist(df_communes_elections_traffic_social_2019["Spotify_srca"], bins=20)
plt.hist(df_communes_elections_traffic_social_2019["Netflix_srca"], bins=20)
plt.hist(df_communes_elections_traffic_social_2019["median_income"], bins=20)
plt.hist(df_communes_elections_traffic_social_2019["unemployment_ratio"], bins=20)
plt.hist(df_communes_elections_traffic_social_2019["pop_0_14"], bins=20)
plt.show()

## Keep Urban & Suburban

In [ ]:
for election in elections:
    df_communes_elections_traffic_social = elections[election][
        "df_communes_elections_traffic_social"
    ]

    df_communes_elections_traffic_social_urban = df_communes_elections_traffic_social[
        df_communes_elections_traffic_social["urbanization_level"] != "rural"
    ]
    elections[election][
        "df_communes_elections_traffic_social_urban"
    ] = df_communes_elections_traffic_social_urban.copy()

df_communes_elections_traffic_social_urban.head(2)

In [ ]:
median_income_column = ["median_income"]
unemployment_ratio_column = ["unemployment_ratio"]
age_columns = [
    "pop_0_14",
    "pop_15_29",
    "pop_30_44",
    "pop_45_59",
    "pop_60_74",
    "pop_75_89",
    "pop_90",
]
social_economical_columns = (
    median_income_column + unemployment_ratio_column + age_columns
)

social_economical_columns_dict_names = {
    "median_income": "Median income",
    "unemployment_ratio": "Unemployment ratio",
    "pop_0_14": "Pop 0-14",
    "pop_15_29": "Pop 15-29",
    "pop_30_44": "Pop 30-44",
    "pop_45_59": "Pop 45-59",
    "pop_60_74": "Pop 60-74",
    "pop_75_89": "Pop 75-89",
    "pop_90": "Pop 90+",
}
social_economical_columns_names = [
    social_economical_columns_dict_names[col] for col in social_economical_columns
]

df_traffic_2019 = elections["europe_2019"]["traffic"]
apps_2019 = df_traffic_2019.columns[1:]
n_apps_2019 = len(apps_2019) // 3  # considering that we have raw, rca and srca traffic
apps_2019_raw = apps_2019[:n_apps_2019]
apps_2019_rca = apps_2019[n_apps_2019 : 2 * n_apps_2019]
apps_2019_srca = apps_2019[2 * n_apps_2019 :]
apps_2019_columns = list(apps_2019_srca)

df_traffic_2024 = elections["europe_2024"]["traffic"]
apps_2024 = df_traffic_2024.columns[1:]
n_apps_2024 = len(apps_2024) // 3  # considering that we have raw, rca and srca traffic
apps_2024_raw = apps_2024[:n_apps_2024]
apps_2024_rca = apps_2024[n_apps_2024 : 2 * n_apps_2024]
apps_2024_srca = apps_2024[2 * n_apps_2024 :]
apps_2024_columns = list(apps_2024_srca)

variables_names = social_economical_columns_dict_names
for col in apps_2019_columns:
    variables_names[col] = col.replace("_srca", "")
for col in apps_2024_columns:
    if col not in variables_names:
        variables_names[col] = col.replace("_srca", "")

## Correlation Matrix | Apps vs Parties

In [ ]:
demographics = ["median_income", "unemployment_ratio"]

social_media = [
    "Facebook_srca",
    "Instagram_srca",
    "LinkedIn_srca",
    "SnapChat_srca",
    "Twitter_srca",
    "Twitch_srca",
    "TikTok_srca",
]

news = [
    "NewsPaper_srca",
    "Sports News_srca",
    "DailyMotion_srca",
    "NewsMag_srca",
    "Google News_srca",
    "TV5MONDE_srca",
]


messaging = [
    "WhatsApp_srca",
    "Apple iMessage_srca",
    "Signal_srca",
    "Discord_srca",
    "Telegram_srca",
]

streamming = [
    "Youtube_srca",
    "Spotify_srca",
    "CanalPlus_srca",
    "Netflix_srca",
    "Apple Music_srca",
    "Disney+_srca",
    "Apple Video_srca",
    "Molotov TV_srca",
    "Pluto TV_srca",
]

rows = demographics + social_media + news + messaging + streamming
row_clusters = [demographics, social_media, news, messaging, streamming]
total_number_of_rows = len(rows)

In [ ]:
def get_correlation_matrix(df, parties_columns):

    correlation_matrix = np.zeros((total_number_of_rows, len(parties_columns)))
    pvalue_matrix = np.zeros_like(correlation_matrix)

    df_columns = df.columns

    for i, row in enumerate(rows):
        for j, party_column in enumerate(parties_columns):

            if row in df_columns:

                correlation, p_value = pearsonr(df[row], df[party_column])
            else:
                correlation = 0
                p_value = 0

            correlation_matrix[i, j] = correlation
            pvalue_matrix[i, j] = p_value

    pvalue_matrix[pvalue_matrix >= 0.05] = 2
    pvalue_matrix[pvalue_matrix < 0.05] = 1
    pvalue_matrix[pvalue_matrix == 2] = 0

    return correlation_matrix, pvalue_matrix

In [ ]:
def plot_correlation_matrix(
    correlation_matrix, pvalue_matrix, row_clusters, main_parties_names, year, filename
):
    fig = plt.figure(figsize=(6, 12))

    my_cmap = matplotlib.colormaps["RdBu_r"].copy()
    my_cmap.set_bad(color="w")
    my_norm = colrs.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

    bottom_count = 0
    row_index = 0
    for row_cluster_index, row_cluster in enumerate(row_clusters):
        n_rows = len(row_cluster)
        fraction = 0.9 * n_rows / total_number_of_rows

        ax = fig.add_axes([0.0, 1 - bottom_count - fraction, 0.9, fraction])
        sub_matrix = correlation_matrix[row_index : row_index + n_rows]
        sub_matrix_pvalue = pvalue_matrix[row_index : row_index + n_rows]
        sub_matrix_significant = sub_matrix / sub_matrix_pvalue

        if year == 2019:
            for app_not_2019 in ["TikTok", "Signal", "Discord", "Pluto TV", "Disney+"]:
                app_not_2019 = f"{app_not_2019}_srca"
                if app_not_2019 in row_cluster:
                    app_not_2019_index = row_cluster.index(app_not_2019)
                    sub_matrix_significant[app_not_2019_index] = np.nan

        heatmap = sns.heatmap(
            sub_matrix_significant[:, :-2],
            cmap=my_cmap,
            norm=my_norm,
            cbar=False,
            annot=True,
            fmt=".2f",
            annot_kws={"fontsize": 15},
        )

        bottom_count += 0.01 + fraction
        row_index += n_rows

        if row_cluster_index == len(row_clusters) - 1:
            heatmap.set_xticks(np.array(range(len(main_parties_names))) + 0.5)
            heatmap.set_xticklabels(
                main_parties_names, rotation=35, fontsize=18, ha="right"
            )
        else:
            heatmap.set_xticks([])

        rows_names = row_clusters[row_cluster_index]
        rows_names = [variables_names[row_name] for row_name in rows_names]

        yticks = np.arange(len(rows_names)) + 0.5
        heatmap.set_yticks(yticks)

        rows_names_ = []
        for name in rows_names:
            if name == "Median income":
                rows_names_.append("Income")
            elif name == "Unemployment ratio":
                rows_names_.append("Unemployment")
            elif name == "Apple iMessage":
                rows_names_.append("iMessage")
            elif name == "NewsPaper":
                rows_names_.append("Online news")
            elif name == "TV5MONDE":
                rows_names_.append("TV5 Monde")
            elif name == "NewsMag":
                rows_names_.append("News Mag.")
            else:
                rows_names_.append(name)

        heatmap.set_yticklabels(rows_names_, rotation=0, fontsize=18)

        if year == 2019:
            for app_not_2019 in ["TikTok", "Signal", "Discord", "Pluto TV", "Disney+"]:
                app_not_2019 = f"{app_not_2019}_srca"
                if app_not_2019 in row_cluster:
                    app_not_2019_index = row_cluster.index(app_not_2019)
                    for i in range(len(main_parties_names)):
                        patch = matplotlib.patches.Rectangle(
                            (i, app_not_2019_index),
                            0.99,
                            0.99,
                            fc="tab:grey",
                            ec="white",
                            alpha=0.1,
                        )
                        heatmap.add_patch(patch)

    plt.savefig(
        f"{IMG_DIR}/correlation_matrix/{filename}.pdf",
        bbox_inches="tight",
        dpi=200,
        transparent=True,
    )
    plt.show()

In [ ]:
fig = plt.figure(figsize=(6, 1))

my_cmap = matplotlib.colormaps["RdBu_r"].copy()
my_cmap.set_bad(color="w")
my_norm = colrs.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

ax = fig.add_axes([0, 0, 0.7, 0.15])
sm = plt.cm.ScalarMappable(cmap=my_cmap, norm=my_norm)
sm.set_array([])
clb = plt.colorbar(sm, cax=ax, orientation="horizontal")
clb.ax.xaxis.set_ticks_position("default")
clb.ax.set_title(rf"Pearson correlation ($\rho$)", fontsize=18)
clb.ax.tick_params(labelsize=18)

plt.savefig(f"{IMG_DIR}/correlation_matrix/colorbar.pdf", bbox_inches="tight")
plt.show()

In [ ]:
for election in elections:

    election_year = elections[election]["date"].year
    print(election_year)

    df_election_results = elections[election]["results"]
    main_parties_columns = list(df_election_results.columns)[
        1:-3
    ]  # remove insee, others, polarization and ideology

    parties_descriptions = elections[election]["parties_descriptions"]
    main_parties_names = [
        parties_descriptions[party]["name"] for party in parties_descriptions
    ]
    main_parties_lr_score = [
        parties_descriptions[party]["lr_score"] for party in parties_descriptions
    ]

    df_election = elections[election]["df_communes_elections_traffic_social"]
    df_election_urban_suburban = df_election[
        df_election["urbanization_level"] != "rural"
    ].copy()
    df_election_rural = df_election[df_election["urbanization_level"] == "rural"].copy()

    areas = {
        "Urban and Suburban Communes": df_election_urban_suburban,
    }

    for area in areas:
        print(f"plotting correlation matrix for {election} in {area}")
        df_election_area = areas[area]

        correlation_matrix, pvalue_matrix = get_correlation_matrix(
            df_election_area, main_parties_columns
        )
        plot_correlation_matrix(
            correlation_matrix,
            pvalue_matrix,
            row_clusters,
            main_parties_names,
            election_year,
            f"{election}_correlation_matrix_{area}",
        )

In [ ]:
def plot_correlation_matrix_horizontal(
    correlation_matrix, pvalue_matrix, row_clusters, main_parties_names, year, filename
):
    fig = plt.figure(figsize=(25, 6))

    my_cmap = matplotlib.colormaps["RdBu_r"].copy()
    my_cmap.set_bad(color="w")
    my_norm = colrs.TwoSlopeNorm(vmin=-1, vcenter=0, vmax=1)

    bottom_count = 0
    row_index = 0
    for row_cluster_index, row_cluster in enumerate(row_clusters):
        n_rows = len(row_cluster)
        fraction = 0.9 * n_rows / total_number_of_rows

        ax = fig.add_axes([bottom_count, 0, fraction, 0.9])

        sub_matrix = correlation_matrix[row_index : row_index + n_rows]
        sub_matrix_pvalue = pvalue_matrix[row_index : row_index + n_rows]
        sub_matrix_significant = sub_matrix / sub_matrix_pvalue

        if year == 2019:
            for app_not_2019 in ["TikTok", "Signal", "Discord", "Pluto TV", "Disney+"]:
                app_not_2019 = f"{app_not_2019}_srca"
                if app_not_2019 in row_cluster:
                    app_not_2019_index = row_cluster.index(app_not_2019)
                    sub_matrix_significant[app_not_2019_index] = np.nan
                    print(app_not_2019_index)


            if row_cluster_index == 1:
                plt.text(
                    fraction + 0.1,
                    -0.3,
                    "Social Media",
                    fontsize=30,
                    ha="left",
                    va="center",
                )
            if row_cluster_index == 2:
                plt.text(fraction, -0.3, "News", fontsize=30, ha="left", va="center")
            if row_cluster_index == 3:
                plt.text(
                    fraction, -0.3, "Messaging", fontsize=30, ha="left", va="center"
                )
            if row_cluster_index == 4:
                plt.text(
                    fraction, -0.3, "Streaming", fontsize=30, ha="left", va="center"
                )

        heatmap = sns.heatmap(
            sub_matrix_significant[:, :-2].T,
            cmap=my_cmap,
            norm=my_norm,
            cbar=False,
            annot=True,
            fmt=".2f",
            annot_kws={"fontsize": 20},
        )

        bottom_count += 0.01 + fraction
        row_index += n_rows

        if row_cluster_index == 0:
            heatmap.set_yticks(np.array(range(len(main_parties_names))) + 0.5)
            heatmap.set_yticklabels(
                main_parties_names, rotation=0, fontsize=30, ha="right"
            )
        else:
            heatmap.set_yticks([])

        rows_names = row_clusters[row_cluster_index]
        rows_names = [variables_names[row_name] for row_name in rows_names]

        rows_names_ = []
        for name in rows_names:
            if name == "Median income":
                rows_names_.append("Income")
            elif name == "Unemployment ratio":
                rows_names_.append("Unemployment")
            elif name == "Apple iMessage":
                rows_names_.append("iMessage")
            elif name == "NewsPaper":
                rows_names_.append("Online news")
            elif name == "TV5MONDE":
                rows_names_.append("TV5 Monde")
            elif name == "NewsMag":
                rows_names_.append("News Mag.")
            else:
                rows_names_.append(name)

        heatmap.set_xticklabels(rows_names_, rotation=45, fontsize=30, ha="right")
        if year == 2019:
            heatmap.set_xticks([])
            heatmap.set_xticklabels([], rotation=0)

        if year == 2019:
            for app_not_2019 in ["TikTok", "Signal", "Discord", "Pluto TV", "Disney+"]:
                app_not_2019 = f"{app_not_2019}_srca"
                if app_not_2019 in row_cluster:
                    app_not_2019_index = row_cluster.index(app_not_2019)
                    for i in range(len(main_parties_names)):
                        patch = matplotlib.patches.Rectangle(
                            (app_not_2019_index + 0.01, i + 0.01),
                            0.98,
                            0.98,
                            fc="tab:grey",
                            ec="white",
                            alpha=0.1,
                        )
                        heatmap.add_patch(patch)

    plt.savefig(
        f"{IMG_DIR}/correlation_matrix/{filename}_horizontal.pdf",
        bbox_inches="tight",
        dpi=200,
        transparent=True,
    )
    plt.show()

In [ ]:
for election in elections:

    election_year = elections[election]["date"].year
    print(election_year)

    df_election_results = elections[election]["results"]
    main_parties_columns = list(df_election_results.columns)[
        1:-3
    ]  # remove insee, others, polarization and ideology

    parties_descriptions = elections[election]["parties_descriptions"]
    main_parties_names = [
        parties_descriptions[party]["name"] for party in parties_descriptions
    ]
    main_parties_lr_score = [
        parties_descriptions[party]["lr_score"] for party in parties_descriptions
    ]

    df_election = elections[election]["df_communes_elections_traffic_social"]
    df_election_urban_suburban = df_election[
        df_election["urbanization_level"] != "rural"
    ].copy()
    df_election_rural = df_election[df_election["urbanization_level"] == "rural"].copy()

    areas = {
        "Urban and Suburban Communes": df_election_urban_suburban,
    }

    for area in areas:
        print(f"plotting correlation matrix for {election} in {area}")
        df_election_area = areas[area]

        correlation_matrix, pvalue_matrix = get_correlation_matrix(
            df_election_area, main_parties_columns
        )
        plot_correlation_matrix_horizontal(
            correlation_matrix,
            pvalue_matrix,
            row_clusters,
            main_parties_names,
            election_year,
            f"{election}_correlation_matrix_{area}",
        )

## VIF

In [ ]:
selected_app = social_media + news + messaging + streamming

In [ ]:
def get_VIF(df, app_columns, election, color):
    VIF_data_app = df[app_columns].copy()

    vifs = [
        variance_inflation_factor(VIF_data_app.values, i)
        for i in range(len(VIF_data_app.columns))
    ]

    vifs_names = zip(VIF_data_app.columns, vifs)
    vifs_names = sorted(vifs_names, key=lambda x: x[1], reverse=True)
    features, vifs = zip(*vifs_names)
    features = [col.replace("_srca", "") for col in features]

    plt.figure(figsize=(20, 5))
    plt.bar(features, vifs, width=0.8, color=color, alpha=0.85)

    print(f"Mean VIF for {election}: {np.mean(vifs):.2f}")
    print(f"Max VIF for {election}: {np.max(vifs):.2f}")

    plt.xticks(rotation=90)
    plt.ylabel("VIF")

    plt.xlim(-1, len(features))
    plt.show()

In [ ]:
apps_columns_2019_ = [col for col in apps_2019_columns if col in selected_app]
get_VIF(
    elections["europe_2019"]["df_communes_elections_traffic_social_urban"],
    apps_columns_2019_,
    "europe_2019",
    "tab:blue",
)

In [ ]:
get_VIF(
    elections["europe_2024"]["df_communes_elections_traffic_social_urban"],
    selected_app,
    "europe_2024",
    "tab:blue",
)

## Save dataset

In [ ]:
for election in elections:

    election_year = elections[election]['date'].year

    df_communes_elections_traffic_social_urban = elections[election]['df_communes_elections_traffic_social_urban']

    df_election_results = elections[election]['results']
    main_parties_columns = list(df_election_results.columns)[1:-3] # remove insee, others, polarization and ideology

    if election_year == 2019:
        apps_columns = apps_2019_columns
        apps_columns = [col for col in apps_columns if col in selected_app]
    elif election_year == 2024:
        apps_columns = apps_2024_columns
        apps_columns = [col for col in apps_columns if col in selected_app]
    else:
        raise ValueError('Year not found')

    df_communes_elections_traffic_social_urban_csv = df_communes_elections_traffic_social_urban[['insee'] + main_parties_columns + ['others_votes'] + ['polarization_dalton', 'ideology'] + social_economical_columns + apps_columns].copy()
    df_communes_elections_traffic_social_urban_csv.to_csv(f'{DATA_DIR}/dirichlet/df_data_{election}_selected.csv', index=False)

    df_communes_elections_traffic_social_all = elections[election]['df_communes_elections_traffic_social']
    df_communes_elections_traffic_social_all_csv = df_communes_elections_traffic_social_all[['insee'] + main_parties_columns + ['others_votes'] + ['polarization_dalton', 'ideology'] + social_economical_columns + apps_columns].copy()
    df_communes_elections_traffic_social_all.to_csv(f'{DATA_DIR}/dirichlet/df_data_{election}_selected_all.csv', index=False)